In [2]:
from pathlib import Path
import json
import re
import pandas as pd
from collections import Counter

FINAL_JSONL = Path("../seed_exports/wiki_mcq_seed_final.jsonl")
REMAINING_MATH_ERRORS_CSV = Path("seed_exports/EDA/wiki_mcq_seed_final_remaining_math_errors.csv")

MANUAL_REVIEW_TSV = Path("seed_exports/EDA/wiki_mcq_remaining_math_manual_review.tsv")
MANUAL_FIXED_TSV = Path("seed_exports/EDA/wiki_mcq_remaining_math_manual_review_fixed.tsv")

OUT_JSONL = Path("../seed_exports/wiki_mcq_seed_final_manual_math_fixed.jsonl")
OUT_FIX_LOG_JSON = Path("seed_exports/EDA/wiki_mcq_remaining_math_manual_fix_log.json")

In [3]:
def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


rows = read_jsonl(FINAL_JSONL)
df_errors = pd.read_csv(REMAINING_MATH_ERRORS_CSV)

df_errors["source_id"] = df_errors["source_id"].astype(str)

print("Rows in final jsonl:", len(rows))
print("Remaining math errors:", len(df_errors))

display(df_errors[["row_idx", "source_id", "subject", "answer", "flags"]].head(30))

Rows in final jsonl: 7929
Remaining math errors: 29


,row_idx,source_id,subject,answer,flags
0,44,machine_learning/test/107,machine_learning,B,"['raw_latex_begin', 'raw_latex_end']"
1,529,high_school_european_history/test/118,high_school_european_history,B,['literal_backslash_n']
2,1022,high_school_european_history/test/83,high_school_european_history,A,['html_entity']
3,1114,machine_learning/test/106,machine_learning,C,['literal_backslash_n']
4,1735,high_school_macroeconomics/test/316,high_school_macroeconomics,C,['unbalanced_math_dollar']
5,2066,high_school_macroeconomics/test/273,high_school_macroeconomics,C,['unbalanced_math_dollar']
6,2347,miscellaneous/test/310,miscellaneous,A,['raw_caret_power']
7,2528,high_school_microeconomics/test/44,high_school_microeconomics,B,['unbalanced_math_dollar']
8,2575,high_school_macroeconomics/test/150,high_school_macroeconomics,C,['unbalanced_math_dollar']
9,2587,high_school_microeconomics/test/32,high_school_microeconomics,D,['unbalanced_math_dollar']


In [4]:
source_id_to_row = {
    str(row["metadata"]["source_id"]): row
    for row in rows
}

review_rows = []

for _, err in df_errors.iterrows():
    source_id = str(err["source_id"])
    row = source_id_to_row[source_id]
    original_text = row["messages"][0]["content"]

    review_rows.append({
        "source_id": source_id,
        "subject": row["metadata"].get("subject"),
        "answer": row["metadata"].get("answer"),
        "flags": err.get("flags", ""),
        "action": "fix",  # fix | keep | drop
        "original_text": original_text,
        "fixed_text": original_text,
        "notes": "",
    })

df_review = pd.DataFrame(review_rows)

MANUAL_REVIEW_TSV.parent.mkdir(parents=True, exist_ok=True)
df_review.to_csv(MANUAL_REVIEW_TSV, sep="\t", index=False)

print("Wrote manual review file:", MANUAL_REVIEW_TSV)
display(df_review[["source_id", "subject", "answer", "flags", "action", "original_text"]].head())

Wrote manual review file: seed_exports/EDA/wiki_mcq_remaining_math_manual_review.tsv


,source_id,subject,answer,flags,action,original_text
0,machine_learning/test/107,machine_learning,B,"['raw_latex_begin', 'raw_latex_end']",fix,Câu hỏi: Câu nào sau đây đúng về hạt nhân tích...
1,high_school_european_history/test/118,high_school_european_history,B,['literal_backslash_n'],fix,Câu hỏi: Câu hỏi này liên quan đến thông tin s...
2,high_school_european_history/test/83,high_school_european_history,A,['html_entity'],fix,Câu hỏi: Câu hỏi này liên quan đến thông tin s...
3,machine_learning/test/106,machine_learning,C,['literal_backslash_n'],fix,Câu hỏi: Giả sử chúng ta có hàm mục tiêu sau: ...
4,high_school_macroeconomics/test/316,high_school_macroeconomics,C,['unbalanced_math_dollar'],fix,Câu hỏi: Cái nào sau đây sẽ có tác động đến GD...
